# 00 — GPT-2 small: the numbers, the shapes, and how it learns

**Series: GPT-2 small (124M) from scratch in pure NumPy, with tiktoken's GPT-2 BPE tokenizer.** Every number in this series is GPT-2 small's real configuration. Where something is too expensive to run at full size on a CPU (for example, checking all 124 million gradients numerically), the notebook says so, explains why, and checks a sample at full size instead.

| # | Notebook | What you build |
|---|---|---|
| **00** | **The numbers and how learning works** | GPT-2 small's config, tokens, every tensor shape, backprop rules, gradient checking |
| 01 | Linear, GELU, MLP | the 768 → 3072 → 768 feed-forward network, forward and backward |
| 02 | Softmax and cross-entropy | the 50,257-way prediction and where the gradient is born |
| 03 | LayerNorm | normalization over 768 features |
| 04 | Causal self-attention | 12 heads × 64 dimensions, forward and backward |
| 05 | Embeddings, blocks, full GPT-2 small | all 124,439,808 parameters, weight tying, full-size gradient check |
| 06 | Training and pretrained weights | the GPT-2 training recipe, real training steps, loading OpenAI's weights into your NumPy model |

Requirements: `numpy`, `tiktoken` (`pip install tiktoken`), and optionally `matplotlib`. The full-size model needs about 0.5 GB of RAM in float32 and about 4 GB for training.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=120)

## 1. GPT-2 small's configuration

These five numbers define the entire architecture:

In [ ]:
config = dict(
    vocab_size = 50257,   # V : number of distinct tokens
    block_size = 1024,    # maximum T : context length in tokens (HF config calls it n_positions / n_ctx)
    n_layer    = 12,      # L : number of transformer blocks
    n_head     = 12,      # H : attention heads per block
    n_embd     = 768,     # C : width of every token vector (d_model)
)
V, n_ctx, L, H, C = (config[k] for k in ["vocab_size", "block_size", "n_layer", "n_head", "n_embd"])
hs = C // H               # head size
print(f"V={V}  block_size={n_ctx}  L={L}  H={H}  C={C}  ->  head size hs={hs}, MLP hidden 4C={4 * C}")

| Symbol | Meaning | GPT-2 small | Where the number comes from |
|---|---|---|---|
| **V** | vocabulary size | **50,257** | 256 byte tokens + 50,000 learned BPE merges + 1 special `<\|endoftext\|>` token (id 50256) |
| **block_size** | max tokens of context | **1,024** | the number of rows in the learned position table `wpe` |
| **L** | transformer blocks | **12** | |
| **H** | attention heads | **12** | |
| **C** | token vector width | **768** | = H × hs = 12 × 64 |
| **hs** | head size | **64** | C / H |
| **4C** | MLP hidden width | **3,072** | the 4× expansion from the original Transformer |
| **B** | sequences per batch | your choice | GPT-2 paper: 512 |
| **T** | tokens per sequence in this batch | ≤ 1,024 | |
| **N** | predictions per batch | B·T | GPT-2: 512 × 1024 = 524,288 tokens per optimizer step |

The larger GPT-2 models use the same code and change only L, H and C: medium (24, 16, 1024), large (36, 20, 1280), XL (48, 25, 1600).

**B, T, C in one line:** a batch is B sequences, each sequence is T **tokens**, and each token is a vector of C = 768 numbers. Throughout this series, T always counts tokens, never characters.

## 2. Tokens: tiktoken's GPT-2 BPE

The model never sees text. It sees integer token ids produced by GPT-2's byte-pair-encoding tokenizer. A token is usually a common word (often including its leading space) or a piece of a rarer word.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print("V =", enc.n_vocab, "  end-of-text token id =", enc.eot_token)

text = ("The quick brown fox jumps over the lazy dog. Language models predict the next token, "
        "and transformers do it for every position at once.")
ids = enc.encode(text)
print(f"{len(text)} characters -> {len(ids)} tokens\n")
print([enc.decode([i]) for i in ids])
print(ids)

Notice:

- `' quick'` is one token, *including* its leading space.
- Common words are one token each; a less common word is split: `' transformers'` → `' transform'` + `'ers'`.
- On typical English text, one token is about 4 characters. So `block_size = 1024` tokens ≈ 700–800 words.
- `<|endoftext|>` (id 50256) is placed between documents in GPT-2's training data, so the model learns where texts begin and end.

## 3. From token stream to a batch: where B and T come from

The model is trained on one task: **given the tokens so far, predict the next token.**

To make a batch from a long stream of token ids: pick B random starting positions, cut out T+1 consecutive tokens at each, and split each chunk into

- `x` = the first T tokens (the input), and
- `y` = the last T tokens (the target), i.e. `x` shifted left by one.

In [ ]:
rng = np.random.default_rng(0)
data = np.array(ids)

def get_batch(data, B, T):
    ix = rng.integers(0, len(data) - T, size=B)          # B random start positions
    x = np.stack([data[i : i + T] for i in ix])           # (B, T)  token ids
    y = np.stack([data[i + 1 : i + T + 1] for i in ix])   # (B, T)  the next token at each position
    return x, y

B, T = 3, 8
x, y = get_batch(data, B, T)
print("x shape", x.shape, " y shape", y.shape, "  <- 3 sequences x 8 TOKENS each\n")
for b in range(B):
    print(f"x[{b}] = {[enc.decode([i]) for i in x[b]]}")
    print(f"y[{b}] = {[enc.decode([i]) for i in y[b]]}\n")

**The key idea: one row of length T is T training examples, not one.** At position `t`, the model must predict token `y[b, t]` having seen only tokens `x[b, 0..t]`:

In [ ]:
b = 0
for t in range(T):
    context = enc.decode(list(x[b, : t + 1]))
    target = enc.decode([y[b, t]])
    print(f"t={t}: context {context!r:45} -> predict {target!r}")

A single forward pass produces predictions at all T positions in parallel. **The causal mask** guarantees that position `t` cannot see positions `> t`; without it, predicting `y[b,t] = x[b,t+1]` would be trivial cheating. So one batch contains **N = B·T** predictions, and the loss is their average. At GPT-2's full context, a single sequence is 1,024 training examples.

## 4. Every tensor in GPT-2 small, with real shapes

Below is the GPT-2 small forward pass with **real dimensions** (C=768, H=12, hs=64, V=50257, L=12) and **real token ids** from two sentences (B=2, T=16). The *weights* are random (std 0.02, as at initialization), because we only want to see shapes here; the real layers come in notebooks 01–05. LayerNorms are left out of this sketch because they don't change shapes.

One fact about NumPy's `@` first: **it treats all leading axes as batch axes and multiplies the last two.**

- `(B, T, 768) @ (768, 3072) → (B, T, 3072)`: the same matrix is applied to each of the B·T token vectors.
- `(B, 12, T, 64) @ (B, 12, 64, T) → (B, 12, T, T)`: an independent `(T,64)@(64,T)` product per sequence and per head.

In [ ]:
r = np.random.default_rng(0)
w = lambda *shape: r.normal(0, 0.02, shape).astype(np.float32)     # random "initialized" weights

sentences = ["The quick brown fox jumps over the lazy dog because it wanted to see the other side",
             "Language models predict the next token, and transformers do it for every position at once"]
T = 16
idx = np.array([enc.encode(s)[:T] for s in sentences])            # (B, T) REAL token ids
B = idx.shape[0]
for row in idx:
    print([enc.decode([i]) for i in row])
print()

wte = w(V, C)                                  # token embedding table (50257, 768)
wpe = w(n_ctx, C)                              # position table        (1024, 768); first T rows used
x = wte[idx] + wpe[np.arange(T)]               # (B,T,768) + (T,768) -> broadcast -> (B, T, 768)
print(f"{'idx':32}{idx.shape}")
print(f"{'x = wte[idx] + wpe[:T]':32}{x.shape}")

for layer in range(L):
    # --- attention sub-block ---
    qkv = x @ w(C, 3 * C)                                         # (B, T, 2304)
    q, k, v = np.split(qkv, 3, axis=-1)                           # each (B, T, 768)
    q = q.reshape(B, T, H, hs).transpose(0, 2, 1, 3)              # (B, 12, T, 64)
    k = k.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
    v = v.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
    att = (q @ k.transpose(0, 1, 3, 2)) / np.float32(np.sqrt(hs)) # (B, 12, T, T)
    att = att + np.triu(np.full((T, T), -np.inf, dtype=np.float32), k=1)
    att = np.exp(att - att.max(-1, keepdims=True)); att /= att.sum(-1, keepdims=True)
    y_ = (att @ v).transpose(0, 2, 1, 3).reshape(B, T, C)          # (B, T, 768) heads concatenated
    x = x + y_ @ w(C, C)                                           # residual add
    # --- MLP sub-block ---
    hid = x @ w(C, 4 * C)                                          # (B, T, 3072)
    x = x + hid @ w(4 * C, C)                                      # (B, T, 768)
    if layer == 0:
        for name, t in [("qkv", qkv), ("q, k, v per head", q), ("attention weights", att),
                        ("heads merged", y_), ("MLP hidden", hid)]:
            print(f"  {name:30}{t.shape}")
        print(f"  ... x {L} blocks ...")
logits = x @ wte.T                                                 # (B, T, 50257): weight tying
print(f"{'logits = x @ wte.T':32}{logits.shape}")

### The same shapes at GPT-2's training size

GPT-2 trains at T = 1024. Here is what the main tensors cost at a micro-batch of B = 8 sequences:

In [ ]:
Bt, Tt = 8, 1024
rows = [("idx (token ids)", (Bt, Tt)),
        ("residual stream x", (Bt, Tt, C)),
        ("qkv", (Bt, Tt, 3 * C)),
        ("attention matrix (per layer)", (Bt, H, Tt, Tt)),
        ("MLP hidden (per layer)", (Bt, Tt, 4 * C)),
        ("logits", (Bt, Tt, V))]
for name, shp in rows:
    print(f"{name:30}{str(shp):24}{np.prod(shp) * 4 / 1e6:>10,.0f} MB in float32")

Two numbers stand out:

- **The attention matrix grows with T².** It is 403 MB *per layer* at T = 1024, and the backward pass needs it, so it is cached for all 12 layers.
- **The logits are the biggest single tensor**: 8 × 1024 × 50257 numbers, 1.6 GB, because every position scores every one of the 50,257 tokens.

This is why GPT-2 was trained on GPUs with a modest micro-batch, using gradient accumulation (notebook 06) to reach 524,288 tokens per optimizer step.

Two structural facts to keep in mind, because they make the backward passes much easier:

1. **Only attention mixes information across positions (the T axis).** Embedding, LayerNorm, Linear, GELU and the MLP all act on each 768-vector independently, so inside them `(B, T, 768)` is just B·T separate vectors and can be reshaped to `(B·T, 768)`.
2. **Sequences in a batch never interact in the forward pass.** They only meet in the loss (an average over them) and in the gradients of shared parameters (a sum over them).

## 5. What "learning" means

Collect every number in every weight matrix, bias, gain and embedding of GPT-2 small into one huge vector $\theta$: **124,439,808 numbers**. The model defines a probability distribution $p_\theta(\text{next token} \mid \text{context})$.

The **loss** on a batch is the average negative log-probability assigned to the correct next tokens:

$$
\mathcal{L}(\theta) = -\frac{1}{N}\sum_{b=1}^{B}\sum_{t=1}^{T} \log p_\theta\big(y_{b,t} \mid x_{b,1..t}\big)
$$

Learning means changing $\theta$ to make $\mathcal{L}$ smaller. In GPT-2's training, one batch held 512 sequences × 1024 tokens, so every optimizer step averaged over $N = 524{,}288$ next-token predictions.

**Why the gradient tells us how.** For a small step $\Delta\theta$, a first-order Taylor expansion gives

$$
\mathcal{L}(\theta + \Delta\theta) \approx \mathcal{L}(\theta) + \nabla_\theta\mathcal{L} \cdot \Delta\theta .
$$

Among all steps of a fixed length, the dot product is most negative when $\Delta\theta$ points exactly opposite to $\nabla_\theta\mathcal{L}$. Hence **gradient descent**:

$$
\theta \leftarrow \theta - \eta\, \nabla_\theta \mathcal{L}
$$

where $\eta$ is the learning rate. GPT-2 uses a smarter variant (AdamW, notebook 06), but the core is the same: **we need $\partial\mathcal{L}/\partial\theta_i$ for every single parameter.** Computing that efficiently is backpropagation.

**The training loop**, which every notebook builds toward:

```
repeat:
    x, y   = get_batch()                 # (B,T), (B,T)
    logits = model.forward(x)            # every layer caches what it needs
    loss   = cross_entropy(logits, y)    # a single number
    zero all gradients
    model.backward(dloss/dlogits)        # fills every parameter's .grad
    clip gradients (optional)
    optimizer.step()                     # theta <- theta - lr * (something based on grad)
```

## 6. Backpropagation: the rules

### 6.1 The one convention

The loss $\mathcal{L}$ is a **scalar**. For *any* tensor $Z$ that appears anywhere in the computation, define

$$
dZ := \frac{\partial \mathcal{L}}{\partial Z}, \qquad \text{with } dZ \text{ having exactly the same shape as } Z.
$$

So `dlogits` has shape `(B,T,V)`, `dW` for a `(C,4C)` weight has shape `(C,4C)`, and so on. Each entry answers: *if I nudge this one number up by ε, how much does the loss change (divided by ε)?*

### 6.2 The chain rule as a vector–Jacobian product

Suppose a layer computes $y = f(x)$, and we already know $dy$. Every $y_i$ may depend on $x_j$, so the chain rule says

$$
dx_j = \sum_i dy_i \, \frac{\partial y_i}{\partial x_j}.
$$

That is $dx = J^\top dy$, where $J$ is the Jacobian. We **never build $J$**: for a `(B,T,C) → (B,T,4C)` layer it would have $(BTC)\times(4BTC)$ entries. Instead, for each layer we derive a cheap closed-form expression for $J^\top dy$. That is what every `backward` method in this series is.

### 6.3 The layer contract

```python
class SomeLayer:
    def forward(self, x):      # compute output, CACHE whatever backward will need
        ...
    def backward(self, dout):  # dout = dL/d(output), same shape as output
        # 1. accumulate dL/d(param) into self.grads[...]   (same shape as each param)
        # 2. return dL/d(input)                             (same shape as input)
```

A whole network's backward is then just: call each layer's `backward` **in reverse order**, passing the returned gradient along. Each layer only needs to know about itself.

### 6.4 Five rules that cover almost everything

1. **Shape rule.** `dZ.shape == Z.shape`, always. If your expression has the wrong shape you have a bug, and the shape usually tells you which transpose or sum you're missing.
2. **Fan-out ⇒ sum.** If a value is used in several places, its gradient is the **sum** of the gradients from each use. This covers:
   - residual connections (`x` goes into both the skip path and the sub-layer);
   - weight tying (the token embedding is used at the input *and* at the output);
   - **broadcasting**: a bias of shape `(C,)` added to `(B,T,C)` is used B·T times, so its gradient is `dout.sum(axis=(0,1))`. Same for any parameter shared across positions or batch elements.
3. **Addition copies the gradient.** If $z = a + b$, then $da = dz$ and $db = dz$. Residual connections are therefore "gradient highways".
4. **Reshape / transpose / split are undone.** Their backward is the inverse reshape, inverse transpose, or concatenation. No arithmetic, just moving numbers back to where they came from.
5. **Elementwise product.** If $z = a \odot b$, then $da = dz \odot b$ and $db = dz \odot a$.

### 6.5 A recipe for deriving any backward pass

1. Write the forward pass with explicit indices, e.g. $y_{ij} = \sum_k x_{ik} W_{kj}$.
2. Compute $\partial\mathcal{L}/\partial(\text{thing}) = \sum (\text{upstream grad}) \times (\text{local derivative})$, summing over every output index that depends on the thing.
3. Turn the index expression back into matrix operations.
4. Check the shapes (rule 1).
5. Verify numerically (section 7).

### 6.6 Why gradients are accumulated with `+=` and zeroed each step

A parameter used in two places (rule 2) receives two contributions during one backward pass. If each `backward` wrote `grad = ...`, the second would overwrite the first. So every layer does `grad += ...`, and the training loop sets all gradients to zero **before** each backward pass. Forgetting to zero is a classic bug: gradients keep growing across steps.

## 7. Gradient checking: how to *prove* a backward pass is right

The definition of a derivative gives a slow but trustworthy way to compute any gradient. The **central difference** is

$$
\frac{\partial \mathcal{L}}{\partial \theta_i} \approx \frac{\mathcal{L}(\theta + \varepsilon e_i) - \mathcal{L}(\theta - \varepsilon e_i)}{2\varepsilon}.
$$

Its error is $O(\varepsilon^2)$, far better than the one-sided $O(\varepsilon)$, because the second-order Taylor terms cancel. Some practical points:

- It needs 2 forward passes **per parameter**. For all of GPT-2 small that would be 2 × 124,439,808 ≈ 249 million forward passes, which is impossible. So this series checks in two ways:
  1. a **complete** check of every gradient on a *tiny* model with the identical architecture (the formulas don't depend on the sizes), and
  2. a **spot check** at full GPT-2 small size: compare a sample of randomly chosen gradient entries against finite differences.
- **Use float64.** With float32 (≈7 significant digits), the subtraction $\mathcal{L}_+ - \mathcal{L}_-$ loses most of its precision. NumPy defaults to float64, so we're fine.
- Compare with the **relative error** $\frac{\lVert a-n\rVert}{\lVert a\rVert+\lVert n\rVert}$. As a rule of thumb, below 1e-7 is correct, around 1e-4 or above is a bug. For single entries deep inside a 12-layer model, up to ~1e-6 is normal (the loss is ~11, and its last digits limit the precision of the difference); a real bug gives ~1e-1 or worse.

**The trick for checking a single layer.** A layer has a tensor output, not a scalar loss. Pick a fixed random tensor $G$ of the output's shape and define the scalar

$$
\mathcal{L} = \sum (\text{out} \odot G).
$$

Then $\partial\mathcal{L}/\partial\,\text{out} = G$ exactly, so `layer.backward(G)` must return $\partial\mathcal{L}/\partial x$, which we can check numerically. A random $G$ exercises every output direction at once.

In [ ]:
def numerical_grad(f, x, eps=1e-6):
    """Central-difference gradient of the scalar function f() with respect to array x.
    x is modified in place (and restored), so f() must read x each time it runs."""
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; f_plus = f()
        x[i] = old - eps; f_minus = f()
        x[i] = old                      # restore!
        grad[i] = (f_plus - f_minus) / (2 * eps)
        it.iternext()
    return grad

def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))

# sanity check on a function whose gradient we know: f(x) = sum(x^3) -> grad = 3x^2
xx = np.random.default_rng(0).normal(size=(3, 4))
print("rel error:", rel_error(numerical_grad(lambda: np.sum(xx**3), xx), 3 * xx**2))

## 8. Everything together on a tiny network

Before GPT, here is the *whole* learning process on a 2-layer network, with every step written by hand. The task is to fit $y = \sin(x)$, and the model is

$$
\text{pred} = \tanh(x W_1 + b_1)\, W_2 + b_2,
\qquad
\mathcal{L} = \frac{1}{N}\sum_n (\text{pred}_n - y_n)^2 .
$$

The derivations of these backward passes:

- **MSE loss.** $\partial\mathcal{L}/\partial\,\text{pred}_n = \frac{2}{N}(\text{pred}_n - y_n)$.
- **tanh.** $\frac{d}{dz}\tanh z = 1-\tanh^2 z$, so `dz = dout * (1 - out**2)`, using the cached output (rule 5).
- **Linear.** The full derivation is in notebook 01: `dW = x.T @ dout`, `db = dout.sum(0)`, `dx = dout @ W.T`.

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, rng):
        self.params = {"W": rng.normal(0, 1 / np.sqrt(n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
    def forward(self, x):
        self.x = x                                   # cache input for backward
        return x @ self.params["W"] + self.params["b"]
    def backward(self, dout):
        self.grads["W"] += self.x.T @ dout           # (n_in, N) @ (N, n_out) -> (n_in, n_out)
        self.grads["b"] += dout.sum(axis=0)          # bias broadcast over N rows -> sum (rule 2)
        return dout @ self.params["W"].T             # (N, n_out) @ (n_out, n_in) -> (N, n_in)

class Tanh:
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out
    def backward(self, dout):
        return dout * (1 - self.out**2)

class MSE:
    def forward(self, pred, y):
        self.diff = pred - y
        return np.mean(self.diff**2)
    def backward(self):
        return 2 * self.diff / self.diff.size

rng = np.random.default_rng(42)
layers = [Linear(1, 16, rng), Tanh(), Linear(16, 1, rng)]
loss_fn = MSE()

def forward(x):
    for layer in layers:
        x = layer.forward(x)
    return x

def backward(dout):
    for layer in reversed(layers):                  # REVERSE order
        dout = layer.backward(dout)

def params():
    return [(p, layer.grads[k]) for layer in layers if hasattr(layer, "params")
            for k, p in layer.params.items()]

def zero_grad():
    for _, g in params():
        g.fill(0.0)                                  # in place! the optimizer holds references

X = rng.uniform(-3, 3, (64, 1))
Y = np.sin(X)

First, prove the backward pass correct, then train.

In [ ]:
zero_grad()
loss_fn.forward(forward(X), Y)
backward(loss_fn.backward())
for p, g in params():
    num = numerical_grad(lambda: loss_fn.forward(forward(X), Y), p)
    print(f"param shape {str(p.shape):10} rel error {rel_error(g, num):.2e}")

In [ ]:
lr = 0.1
for step in range(2001):
    pred = forward(X)                   # 1. forward (caches)
    loss = loss_fn.forward(pred, Y)     # 2. loss
    zero_grad()                         # 3. clear old grads
    backward(loss_fn.backward())        # 4. backward fills grads
    for p, g in params():               # 5. update, IN PLACE
        p -= lr * g
    if step % 400 == 0:
        print(f"step {step:5d}  loss {loss:.5f}")

A subtle but critical detail: the update is `p -= lr * g`, which modifies the array **in place**. Writing `p = p - lr * g` would create a *new* array bound to the local name `p`, and the layer would keep using the old weights, so nothing would learn. The same applies to `g.fill(0.0)` in `zero_grad`. In the GPT code, the optimizer holds references to the very same arrays the layers use, so in-place updates are essential.

## Summary

- **GPT-2 small**: V = 50,257 tokens, context 1,024, 12 blocks, 12 heads × 64 = 768 features, MLP hidden 3,072, 124,439,808 parameters.
- **(B, T, C)**: B sequences × T tokens × 768 features. Most layers process the B·T token vectors independently; only attention mixes along T.
- **One batch = B·T next-token predictions**, made possible in one pass by the causal mask. The loss is their average.
- **Learning = gradient descent** on that loss, so we need $\partial\mathcal{L}/\partial\theta$ for every parameter.
- **Backprop** = the chain rule applied layer by layer in reverse. Each layer turns $d(\text{out})$ into $d(\text{in})$ and accumulates its parameter gradients, using a closed-form vector–Jacobian product.
- **Rules:** same shape; fan-out sums (including broadcast ⇒ sum over B, T); addition copies; reshape and transpose invert; elementwise products swap partners.
- **Always gradient-check** with central differences in float64.

Next: **01 — Linear, GELU, MLP**, where we derive the Linear backward pass for `(B, T, 768)` inputs and build GPT-2's 768 → 3072 → 768 MLP.